# Memory AI Lab — H3 : Enrichissement des Artefacts

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

## Pourquoi

La formule AttachScore est :
```
Score = α·Sem + β·Ent + γ·Temp + δ·Goal - ρ·Age
```

Actuellement :
- **β·Ent = 0** → `artifact.entities = []` sur tous les messages
- **δ·Goal = 0** → `artifact.goal_vector = None` sur tous les messages

Ce notebook active ces deux termes :
1. **Entités** — GLiNER `urchade/gliner_multi-v2.1` (multilingue FR+EN, zero-shot)
2. **Goal vectors** — `goal_heuristics.py` (type + patterns lexicaux + entités → phrase → mE5)

## Sorties sur Drive (`memory_ai_data/`)
```
group_entities_gliner.json    — {idx: [entity_str, ...]} pour chaque message
group_goal_vectors.npy        — (n_msgs, 768) float32
```

Ces fichiers sont chargés dans 01 et 02 pour enrichir les artefacts avant la segmentation.

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!pip install gliner -q
!python -m spacy download fr_core_news_sm -q
print('✓ OK')
print()
print('⚠️  Si première exécution : Exécution > Redémarrer la session,')
print('   puis relancer à partir de la cellule 3.')

In [ ]:
# ── CELLULE 3 : Google Drive + Copie locale ────────────────────────────────
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/memory_ai_data'
LOCAL_DIR = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

for fname in ['group_anon.txt', 'group_embeddings_me5.npy']:
    src, dst = f'{DRIVE_DIR}/{fname}', f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        print(f'  Copie {fname} ...', end=' ', flush=True)
        shutil.copy2(src, dst)
        print('✓')
    elif os.path.exists(dst):
        print(f'  {fname} déjà en local ✓')
    else:
        print(f'  ⚠️  {fname} absent sur Drive')

DATA_DIR = LOCAL_DIR
print(f'\n✓ DATA_DIR = {DATA_DIR}')

In [ ]:
# ── CELLULE 4 : Parse + embeddings baseline ────────────────────────────────
import numpy as np
from pathlib import Path
from parsers.whatsapp_parser import parse_whatsapp_chat

all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
print(f'Messages parsés : {len(all_artifacts)}')

EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings_me5.npy'
all_embeddings = np.load(EMBED_CACHE)
print(f'Embeddings chargés : {all_embeddings.shape}')

# Vérification types d'artefact présents
from collections import Counter
types = Counter(a.artifact_type for a in all_artifacts)
print(f'\nTypes : {dict(types)}')

In [ ]:
# ── CELLULE 5 : Extraction d'entités — GLiNER ──────────────────────────────
# urchade/gliner_multi-v2.1 : multilingue FR+EN, zero-shot
# Durée estimée : ~10 min GPU T4 pour ~11K messages
# Cache sur Drive — skip si déjà calculé
import json
from collections import defaultdict
from entity_extractor import EntityExtractor

ENTITIES_CACHE_LOCAL = f'{DATA_DIR}/group_entities_gliner.json'
ENTITIES_CACHE_DRIVE = f'{DRIVE_DIR}/group_entities_gliner.json'

# Essayer Drive en premier
if not os.path.exists(ENTITIES_CACHE_LOCAL) and os.path.exists(ENTITIES_CACHE_DRIVE):
    shutil.copy2(ENTITIES_CACHE_DRIVE, ENTITIES_CACHE_LOCAL)
    print('Entités copiées depuis Drive')

if os.path.exists(ENTITIES_CACHE_LOCAL):
    with open(ENTITIES_CACHE_LOCAL) as f:
        entities_by_idx = {int(k): v for k, v in json.load(f).items()}
    print(f'✓ Entités chargées depuis cache ({len(entities_by_idx)} artefacts avec entités)')
else:
    print('Extraction GLiNER sur tous les messages...')
    extractor = EntityExtractor(backend='gliner', min_confidence=0.55)

    # Batch par tranches de 500 pour le suivi
    CHUNK = 500
    entities_by_idx = {}   # idx → [surface_form, ...]

    for start in range(0, len(all_artifacts), CHUNK):
        chunk = all_artifacts[start:start + CHUNK]
        mentions = extractor.extract_batch(chunk)

        # Grouper par artefact
        by_id = defaultdict(set)
        for m in mentions:
            # Exclure auteur (metadata) et regex (URL, email, etc.)
            if m.extraction_method in ('gliner', 'spacy'):
                by_id[m.source_artifact_id].add(m.surface_form)

        for local_i, art in enumerate(chunk):
            global_i = start + local_i
            ents = list(by_id.get(art.id, []))
            if ents:
                entities_by_idx[global_i] = ents

        done = min(start + CHUNK, len(all_artifacts))
        n_with = sum(1 for i in range(start, done) if i in entities_by_idx)
        print(f'  [{done}/{len(all_artifacts)}] artefacts avec entités : {n_with}')

    # Sauvegarder
    with open(ENTITIES_CACHE_LOCAL, 'w') as f:
        json.dump({str(k): v for k, v in entities_by_idx.items()}, f, ensure_ascii=False)
    shutil.copy2(ENTITIES_CACHE_LOCAL, ENTITIES_CACHE_DRIVE)
    print(f'\n✓ Entités sauvegardées → Drive:{ENTITIES_CACHE_DRIVE}')

# Assigner aux artefacts
for i, art in enumerate(all_artifacts):
    art.entities = entities_by_idx.get(i, [])

n_enriched = sum(1 for a in all_artifacts if a.entities)
total_ents = sum(len(a.entities) for a in all_artifacts)
print(f'\nArtefacts avec entités : {n_enriched}/{len(all_artifacts)} ({100*n_enriched/len(all_artifacts):.1f}%)')
print(f'Total mentions extraites : {total_ents}')
print(f'Moy par artefact enrichi : {total_ents/max(n_enriched,1):.1f}')

In [ ]:
# ── CELLULE 6 : Inspection qualitative des entités ─────────────────────────
from collections import Counter

# Top entités globales
all_ents_flat = [e for a in all_artifacts for e in a.entities]
top = Counter(all_ents_flat).most_common(20)
print('Top 20 entités extraites :')
for ent, cnt in top:
    print(f'  {cnt:4d}  {ent}')

print()
# Exemples d'artefacts riches
rich = [(i, a) for i, a in enumerate(all_artifacts) if len(a.entities) >= 3]
print(f'Artefacts avec ≥ 3 entités : {len(rich)}')
print('\nExemples :')
import random
random.seed(42)
for i, art in random.sample(rich[:200], min(5, len(rich))):
    print(f'  [{i}] {art.content[:80]}')
    print(f'       → {art.entities}')

In [ ]:
# ── CELLULE 7 : Calcul des goal vectors ───────────────────────────────────
# Entités déjà assignées aux artefacts (cellule 5)
# goal_heuristics.py combine type + patterns lexicaux + entités → phrase → mE5
import torch
from sentence_transformers import SentenceTransformer
from goal_heuristics import compute_goal_vectors

GOAL_CACHE_LOCAL = f'{DATA_DIR}/group_goal_vectors.npy'
GOAL_CACHE_DRIVE = f'{DRIVE_DIR}/group_goal_vectors.npy'

# Essayer Drive en premier
if not os.path.exists(GOAL_CACHE_LOCAL) and os.path.exists(GOAL_CACHE_DRIVE):
    shutil.copy2(GOAL_CACHE_DRIVE, GOAL_CACHE_LOCAL)
    print('Goal vectors copiés depuis Drive')

if os.path.exists(GOAL_CACHE_LOCAL):
    goal_vectors = np.load(GOAL_CACHE_LOCAL)
    assert len(goal_vectors) == len(all_artifacts)
    print(f'✓ Goal vectors chargés depuis cache  shape={goal_vectors.shape}')
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Calcul goal vectors sur {device} ...')

    # Même modèle que le pipeline principal — même espace vectoriel
    MODEL_NAME = 'intfloat/multilingual-e5-base'
    _model = SentenceTransformer(MODEL_NAME, device=device)

    def embed_fn(texts):
        # Pas de prefix 'passage:' — les phrases d'intention sont des requêtes
        return _model.encode(
            ['query: ' + t for t in texts],
            batch_size=256, show_progress_bar=False,
            device=device, convert_to_numpy=True,
        ).astype(np.float32)

    goal_vectors = compute_goal_vectors(all_artifacts, embed_fn)

    np.save(GOAL_CACHE_LOCAL, goal_vectors)
    shutil.copy2(GOAL_CACHE_LOCAL, GOAL_CACHE_DRIVE)
    print(f'✓ Goal vectors sauvegardés → Drive:{GOAL_CACHE_DRIVE}')

# Assigner aux artefacts
for i, art in enumerate(all_artifacts):
    art.goal_vector = goal_vectors[i]

n_nonzero = int((np.linalg.norm(goal_vectors, axis=1) > 0).sum())
print(f'\nShape          : {goal_vectors.shape}')
print(f'Artefacts actifs : {n_nonzero}/{len(all_artifacts)} ({100*n_nonzero/len(all_artifacts):.1f}%)')
print(f'Norme moyenne    : {np.linalg.norm(goal_vectors, axis=1).mean():.4f}')

In [ ]:
# ── CELLULE 8 : Inspection goal vectors ───────────────────────────────────
from goal_heuristics import infer_goal_phrases

# Exemples de phrases générées par heuristique
print('Exemples de goal phrases par artefact :')
print('=' * 60)
sample_idx = [i for i, a in enumerate(all_artifacts) if a.entities][:8]
for i in sample_idx:
    art = all_artifacts[i]
    phrases = infer_goal_phrases(art)
    print(f'\n[{i}] {art.content[:70]}')
    print(f'     entities  : {art.entities[:4]}')
    print(f'     phrases   : {phrases}')

# Distribution des phrases (signal 1 = type uniquement vs multi-signal)
from goal_heuristics import infer_goal_phrases
n_signals = [len(infer_goal_phrases(a)) for a in all_artifacts]
from collections import Counter
dist = Counter(n_signals)
print('\n\nDistribution des signaux par artefact :')
for k in sorted(dist):
    bar = '█' * (dist[k] // 50)
    print(f'  {k} signal(s) : {dist[k]:5d}  {bar}')

## Étapes suivantes — Intégration dans les notebooks 01 et 02

Ajouter après la cellule 4 (parse + embeddings) dans **01_eval_ari.ipynb** et **02_train_boundary_detector.ipynb** :

```python
# ── Chargement des enrichissements (entités + goal vectors) ────────────────
import json, shutil

# Entités GLiNER
ENT_LOCAL = f'{DATA_DIR}/group_entities_gliner.json'
ENT_DRIVE = f'{DRIVE_DIR}/group_entities_gliner.json'
if not os.path.exists(ENT_LOCAL) and os.path.exists(ENT_DRIVE):
    shutil.copy2(ENT_DRIVE, ENT_LOCAL)
if os.path.exists(ENT_LOCAL):
    with open(ENT_LOCAL) as f:
        ents = {int(k): v for k, v in json.load(f).items()}
    for i, art in enumerate(all_artifacts):
        art.entities = ents.get(i, [])
    print(f'✓ Entités chargées')

# Goal vectors
GV_LOCAL = f'{DATA_DIR}/group_goal_vectors.npy'
GV_DRIVE = f'{DRIVE_DIR}/group_goal_vectors.npy'
if not os.path.exists(GV_LOCAL) and os.path.exists(GV_DRIVE):
    shutil.copy2(GV_DRIVE, GV_LOCAL)
if os.path.exists(GV_LOCAL):
    gv = np.load(GV_LOCAL)
    for i, art in enumerate(all_artifacts):
        art.goal_vector = gv[i]
    print(f'✓ Goal vectors chargés  shape={gv.shape}')
```

Ensuite re-entraîner le TCN (02) et re-lancer Optuna (01) pour mesurer l'impact de β·Ent + δ·Goal.